In [2]:
import pandas as pd
import numpy as np
import random

In [5]:
from pathlib import Path
import zipfile

zip_file_path = Path("../data/ipl_male_csv2.zip")
extract_folder = Path("../data/ipl2")

extract_folder.mkdir(parents=True, exist_ok=True)

print("Extracting files...")

with zipfile.ZipFile(zip_file_path, "r") as zip_ref:
    zip_ref.extractall(extract_folder)

print(f"Extraction complete! Files are saved to: {extract_folder}")

Extracting files...
Extraction complete! Files are saved to: ..\data\ipl2


We train a regression model that uses a baseline of 2 * (runs scored in 3 overs) and use these features:


*   Number of wickets
*   Number of dot-balls


*   Number of boundaries
*   Momentum - (runs in 3rd over) - (run rate in 3 overs)

Then we use data only from 2024 onwards to train the model.
Instead of a normal least squares approxiamation, we use a weighted on that gives more preference to the newer years and hence we arrive at our required beta values




In [1]:
import os
import pandas as pd
import numpy as np
from scipy.linalg import lstsq
from pathlib import Path

# ---------------------------------------------------------
# 1. Paths and Initialization
# ---------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
data_path = DATA_DIR / "ipl2"

raw_training_data = []

print("Scanning files and extracting features...")

# ---------------------------------------------------------
# 2. Data Loading & Parsing
# ---------------------------------------------------------
for file in data_path.iterdir():
    # Only process standard match CSVs, skip info files
    if file.suffix == ".csv" and "_info" not in file.name:
        match_file = file

        try:
            # Read the raw CSV file
            df = pd.read_csv(match_file, header=None, engine="python", on_bad_lines="skip")

            # Check if the file has a header row natively
            if str(df.iloc[0, 0]).strip().lower() == "match_id":
                df.columns = df.iloc[0].astype(str).str.strip().str.lower()
                df = df[1:].reset_index(drop=True)
            else:
                # Fallback: Apply expected base columns if headers are missing
                base_cols = [
                    "match_id", "season", "start_date", "venue", "innings", "ball", 
                    "ball_2", "batting_team", "bowling_team", "striker", "non_striker", 
                    "bowler", "runs_off_bat", "extras", "wides", "noballs", "byes", 
                    "legbyes", "penalty", "wicket_type", "player_out"
                ]
                
                col_names = base_cols[:len(df.columns)]
                
                # If the dataframe has more columns than expected, append generic names
                if len(df.columns) > len(base_cols):
                    col_names += [f"col_{i}" for i in range(len(base_cols), len(df.columns))]
                
                df.columns = col_names

        except Exception:
            # Skip unreadable files
            continue

        # ---------------------------------------------------------
        # 3. Data Cleaning
        # ---------------------------------------------------------
        # Clean up the 'season' format (e.g., "2020/21" -> "2020") and convert to numeric
        df["season"] = pd.to_numeric(df["season"].astype(str).str.split("/").str[0], errors="coerce")
        df["innings"] = pd.to_numeric(df["innings"], errors="coerce")
        
        # Convert runs and fill NaNs with 0
        df["runs_off_bat"] = pd.to_numeric(df["runs_off_bat"], errors="coerce").fillna(0)
        df["extras"] = pd.to_numeric(df["extras"], errors="coerce").fillna(0)

        # Drop rows missing crucial metadata
        df = df.dropna(subset=["season", "innings", "ball", "venue"])

        if len(df) == 0:
            continue

        season = int(df["season"].iloc[0])
        venue = str(df["venue"].iloc[0]).split(",")[0].strip()

        # Isolate training set strictly to seasons prior to 2023
        if season >= 2023:
            continue

        # Extract the current over number from the 'ball' column (e.g., ball 2.4 -> over 2)
        if "over" not in df.columns:
            df["over"] = df["ball"].astype(str).str.split(".").str[0].astype(int)

        # ---------------------------------------------------------
        # 4. Feature Engineering (Per Inning)
        # ---------------------------------------------------------
        for inning in [1, 2]:
            df_inn = df[df["innings"] == inning]

            if len(df_inn) == 0:
                continue

            # Flag if the team is chasing (2nd innings)
            is_chasing = 1 if inning == 2 else 0

            # Filter data for the first 3 overs (overs 0, 1, 2)
            df_3 = df_inn[df_inn["over"] < 3]

            if len(df_3) == 0:
                continue

            # R3: Total runs scored in the first 3 overs
            R3 = (df_3["runs_off_bat"] + df_3["extras"]).sum()

            # W3: Total wickets lost in the first 3 overs
            W3 = df_3["wicket_type"].notna().sum() if "wicket_type" in df_3.columns else 0

            # Momentum: Difference between runs in the 3rd over and the average of the first two
            last_over = df_3[df_3["over"] == 2]
            last_runs = (last_over["runs_off_bat"] + last_over["extras"]).sum()
            momentum = last_runs - (R3 / 3.0)

            # Dot balls and Boundaries in the first 3 overs
            dot_balls = ((df_3["runs_off_bat"] == 0) & (df_3["extras"] == 0)).sum()
            boundaries = (df_3["runs_off_bat"] >= 4).sum()

            # Target Variable Y6: Total runs scored in the Powerplay (first 6 overs)
            df_6 = df_inn[df_inn["over"] < 6]
            Y6 = (df_6["runs_off_bat"] + df_6["extras"]).sum()

            # Store the extracted features for this inning
            raw_training_data.append({
                "W3": W3,
                "momentum": momentum,
                "dot_balls": dot_balls,
                "boundaries": boundaries,
                "is_chasing": is_chasing,
                "venue": venue,
                "Y6": Y6,
                "R3": R3,
                "season": season,
            })

# Safety check
if not raw_training_data:
    raise ValueError(f"No training samples found. CSV files detected: {len(list(data_path.glob('*.csv')))}")

# ---------------------------------------------------------
# 5. Venue Encoding (One-Hot)
# ---------------------------------------------------------
all_venues = sorted(list(set(d["venue"] for d in raw_training_data)))

# Drop the first venue to use it as the baseline (avoids the dummy variable trap)
baseline_venue = all_venues[0]
unique_venues = all_venues[1:]

print(f"Found {len(all_venues)} unique venues.")
print(f"Baseline venue: {baseline_venue}")

# ---------------------------------------------------------
# 6. Building the Design Matrix (X) and Target Array (Y)
# ---------------------------------------------------------
X_list, Y_list, R3_list, weights = [], [], [], []

for d in raw_training_data:
    # Base features (1 is for the intercept)
    row = [
        1, 
        d["W3"], 
        d["momentum"], 
        d["dot_balls"], 
        d["boundaries"], 
        d["is_chasing"]
    ]

    # Append one-hot encoded venue flags
    venue_vec = [1 if v == d["venue"] else 0 for v in unique_venues]
    row.extend(venue_vec)

    X_list.append(row)
    Y_list.append(d["Y6"])
    R3_list.append(d["R3"])
    weights.append(d["season"])

X = np.array(X_list)
Y = np.array(Y_list)
R3_arr = np.array(R3_list)
weights = np.array(weights, dtype=float)

# Calculate naive projection (double the 3-over score)
baseline = 2 * R3_arr

# We train the model to predict the *residual* (how many runs above/below the naive projection)
Y = Y - baseline  

print(f"Training Samples Built: {len(X)}")

# ---------------------------------------------------------
# 7. Time-weighted Ridge Regression Modeling
# ---------------------------------------------------------
# Apply exponential decay weighting so recent seasons impact the model more heavily
weights = np.exp((weights - weights.max()) / 1.5)
W_sqrt = np.sqrt(weights)

# Scale X and Y by the square root of the weights
X_w = X * W_sqrt[:, None]
Y_w = Y * W_sqrt

# Ridge Regression setup (L2 regularization to prevent overfitting on venues with sparse data)
lambda_ridge = 15.0
penalty_matrix = np.sqrt(lambda_ridge) * np.eye(X_w.shape[1])

# Do not penalize the intercept (index 0)
penalty_matrix[0, 0] = 0

# Augment matrices for Ridge solve via ordinary least squares (OLS)
X_aug = np.vstack([X_w, penalty_matrix])
Y_aug = np.concatenate([Y_w, np.zeros(X_w.shape[1])])

# Solve the least squares problem: min ||X_aug * beta - Y_aug||_2
beta, residuals, rank, s = lstsq(X_aug, Y_aug)

print("Model trained successfully!")
print(f"Beta length: {len(beta)}")

Scanning files and extracting features...
Found 39 unique venues.
Baseline venue: Arun Jaitley Stadium
Training Samples Built: 1898
Model trained successfully!
Beta length: 44


Using data from a particular file here we can check our result by multiplying the required feature vector with values from the match with beta to obtain our estimated 6 over score

In [2]:
def predict_from_file(match_file, beta, unique_venues, chosen_inning):
    """
    Reads a match CSV file, extracts first-3-overs features for a specific inning,
    and uses the trained Ridge regression weights (beta) to predict Powerplay runs.
    """
    
    # ---------------------------------------------------------
    # 1. Load and Parse Data
    # ---------------------------------------------------------
    df = pd.read_csv(match_file, header=None, engine="python", on_bad_lines="skip")

    # Check if headers exist, otherwise apply base schema
    if str(df.iloc[0, 0]).strip().lower() == "match_id":
        df.columns = df.iloc[0].astype(str).str.strip().str.lower()
        df = df[1:].reset_index(drop=True)
    else:
        base_cols = [
            "match_id", "season", "start_date", "venue", "innings",
            "ball", "ball_2", "batting_team", "bowling_team",
            "striker", "non_striker", "bowler",
            "runs_off_bat", "extras", "wides", "noballs",
            "byes", "legbyes", "penalty", "wicket_type", "player_out"
        ]

        col_names = base_cols[:len(df.columns)]

        # Handle unexpected extra columns gracefully
        if len(df.columns) > len(base_cols):
            col_names += [f"col_{i}" for i in range(len(base_cols), len(df.columns))]

        df.columns = col_names

    # Normalize legacy column names for compatibility
    if "runs_off_bat" not in df.columns and "batsman_runs" in df.columns:
        df = df.rename(columns={"batsman_runs": "runs_off_bat"})

    if "extras" not in df.columns and "extra_runs" in df.columns:
        df = df.rename(columns={"extra_runs": "extras"})

    # Sanity check: Ensure critical data exists before proceeding
    if "runs_off_bat" not in df.columns or "venue" not in df.columns:
        return None, None

    # ---------------------------------------------------------
    # 2. Data Cleaning
    # ---------------------------------------------------------
    df["runs_off_bat"] = pd.to_numeric(df["runs_off_bat"], errors="coerce").fillna(0)
    df["extras"] = pd.to_numeric(df["extras"], errors="coerce").fillna(0)
    df["innings"] = pd.to_numeric(df["innings"], errors="coerce")

    df = df.dropna(subset=["innings", "ball", "venue"])

    # Extract current over from the 'ball' column
    if "over" not in df.columns:
        df["over"] = df["ball"].astype(str).str.split(".").str[0].astype(int)

    # ---------------------------------------------------------
    # 3. Filter by Chosen Inning
    # ---------------------------------------------------------
    valid_innings = [i for i in df["innings"].unique() if i in [1, 2]]

    if chosen_inning not in valid_innings:
        return None, None

    df = df[df["innings"] == chosen_inning]

    # ---------------------------------------------------------
    # 4. Feature Extraction (First 3 Overs)
    # ---------------------------------------------------------
    is_chasing = 1 if chosen_inning == 2 else 0
    venue = str(df["venue"].iloc[0]).split(",")[0].strip()

    df_3 = df[df["over"] < 3]

    # Cumulative runs and wickets in the first 3 overs
    R3 = (df_3["runs_off_bat"] + df_3["extras"]).sum()
    W3 = df_3["wicket_type"].notna().sum() if "wicket_type" in df_3.columns else 0

    # Momentum: Difference between the 3rd over's runs and the average of the first two
    last_over = df_3[df_3["over"] == 2]
    last_runs = (last_over["runs_off_bat"] + last_over["extras"]).sum()
    momentum = last_runs - (R3 / 3.0)

    # Count dot balls and boundaries
    dot_balls = ((df_3["runs_off_bat"] == 0) & (df_3["extras"] == 0)).sum()
    boundaries = (df_3["runs_off_bat"] >= 4).sum()

    # ---------------------------------------------------------
    # 5. Build Feature Vector & Predict
    # ---------------------------------------------------------
    # Base feature array matching training design matrix
    row = [
        1,  # Intercept
        W3,
        momentum,
        dot_balls,
        boundaries,
        is_chasing
    ]

    # Apply one-hot venue encoding identical to training
    venue_vec = [1 if v == venue else 0 for v in unique_venues]
    row.extend(venue_vec)

    x = np.array(row)

    # Generate Prediction: 
    # Since the model was trained on the residual (Y - 2*R3), 
    # we add 2*R3 back to the dot product to get the actual projected score.
    y_pred = float(np.dot(x, beta) + 2 * R3)

    # Extract Ground Truth: Actual runs scored in the Powerplay (first 6 overs)
    df_6 = df[df["over"] < 6]
    y_true = (df_6["runs_off_bat"] + df_6["extras"]).sum()

    return y_pred, y_true

We use random matches from 2026 to check our rmse values in the next 2 boxes

In [3]:
from pathlib import Path
import pandas as pd

# ---------------------------------------------------------
# 1. Setup and Initialization
# ---------------------------------------------------------
data_path = Path("../data/ipl2")
valid_test_files = []

print("Scanning for test matches (2023 onwards)...")

# ---------------------------------------------------------
# 2. File Scanning & Filtering
# ---------------------------------------------------------
if data_path.exists():
    for file in data_path.iterdir():
        
        # Only process CSV files with purely numeric names (e.g., "1359475.csv")
        # This naturally excludes metadata files like "1359475_info.csv"
        if file.suffix == ".csv" and file.stem.isdigit():
            try:
                # Memory optimization: Read only the first 2 rows to find the season
                df_temp = pd.read_csv(
                    file,
                    header=None,
                    nrows=2,
                    engine="python",
                    on_bad_lines="skip"
                )

                # Check if the first row is a header string
                if str(df_temp.iloc[0, 0]).strip().lower() == "match_id":
                    # Data is in the second row (index 1), 'season' is usually column 1
                    season_raw = str(df_temp.iloc[1, 1]).split("/")[0]
                else:
                    # No header; data starts on the first row (index 0)
                    season_raw = str(df_temp.iloc[0, 1]).split("/")[0]

                # Convert the extracted season string to an integer
                season = int(pd.to_numeric(season_raw, errors="coerce"))

                # Append files from 2023 or later to the validation/test set
                if season >= 2023:
                    valid_test_files.append(file)

            except Exception:
                # Gracefully skip files that are corrupt or lack proper season data
                continue

# ---------------------------------------------------------
# 3. Results Output
# ---------------------------------------------------------
print(f"Test matches found: {len(valid_test_files)}")

Scanning for test matches (2023 onwards)...
Test matches found: 293


In [4]:
import numpy as np

# ---------------------------------------------------------
# 1. Model Evaluation (Test Set)
# ---------------------------------------------------------
errors = []

print("Evaluating model on test matches...")

# Iterate through all valid test match files (2023 onwards)
for match_file in valid_test_files:
    
    # Evaluate both the 1st and 2nd innings for each match
    for inning in [1, 2]:
        
        # Predict Powerplay runs using our trained model (beta)
        y_pred, y_true = predict_from_file(match_file, beta, unique_venues, inning)
        
        # If valid data was returned, calculate the squared error
        if y_pred is not None and y_true is not None:
            squared_error = (y_pred - y_true) ** 2
            errors.append(squared_error)

# ---------------------------------------------------------
# 2. Final Metric Calculation (RMSE)
# ---------------------------------------------------------
# Root Mean Squared Error (RMSE) represents the average error of our 
# predictions in the actual unit (Runs). 
rmse = np.sqrt(np.mean(errors))

print("=" * 40)
print(f"Test innings evaluated: {len(errors)}")
print(f"Final RMSE: {rmse:.2f} runs")

Evaluating model on test matches...
Test innings evaluated: 582
Final RMSE: 10.34 runs


As an improvement, we can try using additional features or adding priors using team info or even batsmen and bowler info